# OmniFile AI Processor - Cloud Notebook

معالج ملفات الذكاء الاصطناعي - دفتر العمل السحابي

---

## الوصف

دفتر ملاحظات شامل لمعالجة الملفات باستخدام الذكاء الاصطناعي على Google Colab. يدعم:

- **استخراج النص** من ملفات PDF والصور باستخدام PyMuPDF و EasyOCR
- **التعرف على اللغة** تلقائياً باستخدام مكتبة langdetect
- **دعم اللغة العربية** عبر Tesseract OCR
- **واجهة تفاعلية** عبر Streamlit مع ngrok
- **تخزين البيانات** في قاعدة بيانات SQLite على Google Drive

### المتطلبات
- حساب Google مع Google Drive
- اتصال بالإنترنت
- متصفح ويب حديث

In [ ]:
# ============================================
# Cell 1: Install System & Python Packages
# ============================================

print("🔧 Installing system packages (poppler, tesseract, Arabic OCR)...")
!apt-get -y install poppler-utils tesseract-ocr tesseract-ocr-ara -qq

print("📦 Installing Python packages...")
!pip install -q PyMuPDF easyocr langdetect streamlit pyngrok filelock pandas Pillow

# --------------------------------------------
# Check GPU availability
# --------------------------------------------
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    print(f"\n✅ GPU Available: {gpu_name}")
    print(f"   VRAM: {gpu_mem:.1f} GB")
else:
    print("\n⚠️ No GPU detected — running on CPU (OCR will be slower).")

print("\n✅ All dependencies installed successfully!")

In [ ]:
# ============================================
# Cell 2: Mount Google Drive & Setup Paths
# ============================================

from google.colab import drive
import os
import sys

# Mount Google Drive
print("📁 Mounting Google Drive...")
drive.mount('/content/drive')

# Project root on Drive
PROJECT_ROOT = '/content/drive/MyDrive/OmniFile_AI'

# Create project directory structure
DIRS = [
    PROJECT_ROOT,
    os.path.join(PROJECT_ROOT, 'uploads'),
    os.path.join(PROJECT_ROOT, 'processed'),
    os.path.join(PROJECT_ROOT, 'outputs'),
    os.path.join(PROJECT_ROOT, 'database'),
    os.path.join(PROJECT_ROOT, 'models'),
    os.path.join(PROJECT_ROOT, 'logs'),
]

for d in DIRS:
    os.makedirs(d, exist_ok=True)
    print(f"  ✅ {d}")

# Set paths as environment variables
os.environ['PROJECT_ROOT'] = PROJECT_ROOT
os.environ['UPLOAD_DIR'] = os.path.join(PROJECT_ROOT, 'uploads')
os.environ['OUTPUT_DIR'] = os.path.join(PROJECT_ROOT, 'outputs')
os.environ['DB_PATH'] = os.path.join(PROJECT_ROOT, 'database', 'omnifile.db')

# Add project root to sys.path
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"\n📂 Project Root: {PROJECT_ROOT}")
print(f"💾 Database Path: {os.environ['DB_PATH']}")
print("\n✅ Google Drive mounted and directories created!")

In [ ]:
# ============================================
# Cell 3: Create / Connect SQLite Database
# ============================================

import sqlite3
from datetime import datetime

DB_PATH = os.environ.get('DB_PATH', '/content/drive/MyDrive/OmniFile_AI/database/omnifile.db')

def get_connection():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA foreign_keys=ON")
    return conn

def init_db():
    conn = get_connection()
    cursor = conn.cursor()

    # Files table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS files (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            filename TEXT NOT NULL,
            filepath TEXT NOT NULL,
            file_type TEXT NOT NULL,
            file_size INTEGER DEFAULT 0,
            upload_time TEXT DEFAULT CURRENT_TIMESTAMP,
            status TEXT DEFAULT 'uploaded',
            processed_time TEXT,
            error_message TEXT
        )
    """)

    # Extracted text table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS extracted_text (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_id INTEGER NOT NULL,
            page_number INTEGER,
            text_content TEXT,
            confidence REAL,
            language TEXT,
            extraction_method TEXT,
            extracted_at TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (file_id) REFERENCES files(id) ON DELETE CASCADE
        )
    """)

    # Processing logs table
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS processing_logs (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            file_id INTEGER,
            action TEXT NOT NULL,
            details TEXT,
            timestamp TEXT DEFAULT CURRENT_TIMESTAMP,
            FOREIGN KEY (file_id) REFERENCES files(id) ON DELETE SET NULL
        )
    """)

    conn.commit()
    conn.close()
    print(f"✅ Database initialized at: {DB_PATH}")

# Initialize the database
init_db()

# Verify by listing tables
conn = get_connection()
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name")
tables = [row['name'] for row in cursor.fetchall()]
conn.close()

print(f"📋 Tables created: {tables}")
print(f"💾 Database file size: {os.path.getsize(DB_PATH) / 1024:.1f} KB")

In [ ]:
# ============================================
# Cell 4: Process PDF with PyMuPDF
# ============================================

import fitz  # PyMuPDF
from langdetect import detect

pdf_path = '/content/drive/MyDrive/python notes.pdf'

# Verify file exists
if not os.path.exists(pdf_path):
    print(f"⚠️ File not found: {pdf_path}")
    print("   Please upload 'python notes.pdf' to your Google Drive root.")
else:
    print(f"📄 Opening: {pdf_path}")
    doc = fitz.open(pdf_path)

    print(f"   Pages: {doc.page_count}")
    print(f"   Metadata: {doc.metadata}")
    print()

    # Extract text from first 3 pages
    max_pages = min(3, doc.page_count)
    all_text = ""

    for page_num in range(max_pages):
        page = doc[page_num]
        text = page.get_text("text")
        all_text += text

        # Detect language
        try:
            lang = detect(text[:500]) if text.strip() else "unknown"
        except Exception:
            lang = "unknown"

        word_count = len(text.split())
        print(f"   📃 Page {page_num + 1}: {word_count} words | Language: {lang}")
        print(f"   {'─' * 60}")
        # Show first 200 chars of each page
        preview = text.strip()[:200]
        print(f"   {preview}...")
        print()

    # Save extracted text to output directory
    output_file = os.path.join(os.environ['OUTPUT_DIR'], 'python_notes_extracted.txt')
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(all_text)

    print(f"💾 Extracted text saved to: {output_file}")
    print(f"📊 Total words extracted: {len(all_text.split())}")
    print(f"📊 Total characters: {len(all_text)}")

    # Store in database
    conn = get_connection()
    cursor = conn.cursor()

    cursor.execute("""
        INSERT INTO files (filename, filepath, file_type, file_size, status, processed_time)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (
        os.path.basename(pdf_path),
        pdf_path,
        'application/pdf',
        os.path.getsize(pdf_path),
        'completed',
        datetime.now().isoformat()
    ))
    file_id = cursor.lastrowid

    for page_num in range(max_pages):
        page = doc[page_num]
        text = page.get_text("text")
        try:
            lang = detect(text[:500]) if text.strip() else "unknown"
        except Exception:
            lang = "unknown"

        cursor.execute("""
            INSERT INTO extracted_text (file_id, page_number, text_content, language, extraction_method)
            VALUES (?, ?, ?, ?, ?)
        """, (file_id, page_num + 1, text, lang, 'PyMuPDF'))

    conn.commit()
    conn.close()

    print(f"🗃️  Stored in database with file_id: {file_id}")
    print("\n✅ PDF processing complete!")

    doc.close()

In [ ]:
# ============================================
# Cell 5: Launch Streamlit App with ngrok
# ============================================

import subprocess
import time
from pyngrok import ngrok

# Kill any existing ngrok tunnels
ngrok.kill()
time.sleep(1)

# Write the Streamlit app to disk
APP_CODE = '''
import streamlit as st
import os
import fitz
import pandas as pd
from langdetect import detect
from datetime import datetime

st.set_page_config(page_title="OmniFile AI Processor", page_icon="📄", layout="wide")

PROJECT_ROOT = os.environ.get("PROJECT_ROOT", "/content/drive/MyDrive/OmniFile_AI")
UPLOAD_DIR = os.environ.get("UPLOAD_DIR", os.path.join(PROJECT_ROOT, "uploads"))
OUTPUT_DIR = os.environ.get("OUTPUT_DIR", os.path.join(PROJECT_ROOT, "outputs"))
DB_PATH = os.environ.get("DB_PATH", os.path.join(PROJECT_ROOT, "database", "omnifile.db"))

st.title("📄 OmniFile AI Processor")
st.caption("معالج ملفات الذكاء الاصطناعي")

tab1, tab2, tab3 = st.tabs(["📤 Upload", "🔍 Process", "📊 Database"])

with tab1:
    st.header("Upload Files")
    uploaded = st.file_uploader("Choose PDF or image files", type=["pdf", "png", "jpg", "jpeg"])
    if uploaded:
        save_path = os.path.join(UPLOAD_DIR, uploaded.name)
        with open(save_path, "wb") as f:
            f.write(uploaded.getbuffer())
        st.success(f"Saved: {uploaded.name} ({uploaded.size} bytes)")

with tab2:
    st.header("Process Files")
    if os.path.exists(UPLOAD_DIR):
        files = os.listdir(UPLOAD_DIR)
        if files:
            selected = st.selectbox("Select a file", files)
            if st.button("Extract Text"):
                filepath = os.path.join(UPLOAD_DIR, selected)
                if filepath.endswith(".pdf"):
                    doc = fitz.open(filepath)
                    full_text = ""
                    for page in doc:
                        full_text += page.get_text("text")
                    doc.close()
                    st.text_area("Extracted Text", full_text, height=400)
                else:
                    st.info("Image OCR not yet available in this demo.")
        else:
            st.info("No files uploaded yet.")

with tab3:
    st.header("Database Records")
    if os.path.exists(DB_PATH):
        import sqlite3
        conn = sqlite3.connect(DB_PATH)
        df = pd.read_sql_query("SELECT * FROM files ORDER BY id DESC", conn)
        conn.close()
        st.dataframe(df, use_container_width=True)
    else:
        st.warning("Database not found. Run the database cell first.")
'''

app_path = '/content/app.py'
with open(app_path, 'w', encoding='utf-8') as f:
    f.write(APP_CODE)

print("📝 Streamlit app written to /content/app.py")

# Start Streamlit in background
process = subprocess.Popen(
    ['streamlit', 'run', 'app.py', '--server.port', '8501',
     '--server.headless', 'true', '--server.enableCORS', 'false'],
    cwd='/content',
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print("⏳ Waiting for Streamlit to start...")
time.sleep(5)

# Create ngrok tunnel
public_url = ngrok.connect(8501)
print(f"\n🎉 Streamlit app is live!")
print(f"🌐 Public URL: {public_url}")
print(f"\n⚠️  Click the URL above to open the app in a new tab.")
print(f"   The tunnel will remain active as long as this cell is running.")
print(f"   To stop: Runtime → Interrupt execution, or run: ngrok.kill()")

# الخطوات التالية — Next Steps

---

## ✅ ما تم إنجازه / What's Done

1. ✅ تثبيت جميع الحزم المطلوبة (PyMuPDF, EasyOCR, Streamlit, ngrok, ...)
2. ✅ ربط Google Drive وإنشاء هيكل المجلدات
3. ✅ إنشاء قاعدة بيانات SQLite مع الجداول الأساسية
4. ✅ معالجة ملف PDF واستخراج النص
5. ✅ إطلاق واجهة Streamlit التفاعلية عبر ngrok

## 🚀 خطوات إضافية / Additional Steps

### إضافة OCR للصور (Image OCR)
```python
import easyocr
reader = easyocr.Reader(['en', 'ar'], gpu=True)
results = reader.readtext('/path/to/image.png')
```

### معالجة ملفات متعددة (Batch Processing)
```python
import glob
for pdf_file in glob.glob('/content/drive/MyDrive/OmniFile_AI/uploads/*.pdf'):
    # Process each file
    pass
```

### تصدير البيانات (Export Data)
```python
import pandas as pd
import sqlite3
conn = sqlite3.connect(DB_PATH)
df = pd.read_sql_query("SELECT * FROM files JOIN extracted_text ON files.id = extracted_text.file_id", conn)
df.to_csv('/content/drive/MyDrive/OmniFile_AI/outputs/export.csv', index=False)
```

## 📌 ملاحظات مهمة / Important Notes

- **مدة الجلسة**: جلسة Colab تنتهي بعد 12 ساعة من عدم النشاط. احفظ عملك بانتظام.
- **الذاكرة**: RAM محدود. لا تعالج ملفات كبيرة جداً في وقت واحد.
- **ngrok**: النفق المجاني يتغير عند كل إعادة تشغيل.
- **Google Drive**: التغييرات تُحفظ تلقائياً في Drive.